# Hybrid Clause and Vague Term Extraction

Working on this approach here, the goal is to extract the contractual clauses and the underlying vague terms in the particular clauses. For this, we will be used 3 datasets:

*   [CUAD](https://huggingface.co/datasets/theatticusproject/cuad)
*   [ContractNLI](https://huggingface.co/datasets/kiddothe2b/contract-nli); extensive information can be found [here](https://stanfordnlp.github.io/contract-nli/). Refer to the paper [here](https://arxiv.org/pdf/2110.01799)
*   [LEDGAR](https://huggingface.co/datasets/coastalcph/lex_glue) (the subset in LexGLUE)

Extracting the anchor clauses and the vague terms will form the first two parts of the [triplet dataset](https://github.com/adrinorosario/legal-pragmatic-inference/blob/main/docs/research/dataset_construction.md).




In [1]:
from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap;       /* CSS3 */
            white-space: -moz-pre-wrap;  /* Mozilla, since 1999 */
            white-space: -pre-wrap;      /* Opera 4-6 */
            white-space: -o-pre-wrap;    /* Opera 7 */
            word-wrap: break-word;       /* Internet Explorer 5.5+ */
        }
    </style>
    '''))

# Swapped 'pre_run' with 'pre_execute' to match modern IPython specifications
get_ipython().events.register('pre_execute', set_css)

In [2]:
import json
import requests
import re
import copy

import torch
from tqdm.auto import tqdm

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_READ_DATASETS_TOKEN")

# Log in to Hugging Face
login(token=hf_token)


## Extraction from CUAD

For this, we are using the JSON file from the HuggingFace page of the CUAD Dataset which can be found [here](https://huggingface.co/datasets/theatticusproject/cuad/tree/main/CUAD_v1)

In [4]:
# read the json file
cuad_v1_json = "/kaggle/input/datasets/adrinorosario/cuad-v1/CUAD_v1.json"

with open(cuad_v1_json, "r") as file:
  data = json.load(file)

data.keys()

dict_keys(['version', 'data'])

In [5]:
len(data["data"]) # contains 510 entries

510

In [6]:
cuad_data = data["data"]
cuad_data[0].keys() # each entry, i.e., a contract contains the title and the paragraphs in it

dict_keys(['title', 'paragraphs'])

In [7]:
print(f"Type of cuad_data[0]['paragraphs']: {type(cuad_data[0]["paragraphs"])}")
print(f"Length of cuad_data[0]['paragraphs]: {len(cuad_data[0]["paragraphs"])}")

print(f"\nType of cuad_data[0]['paragraphs'][0]: {type(cuad_data[0]['paragraphs'][0])}")
print(f"Length of cuad_data[0]['paragraphs'][0]: {len(cuad_data[0]['paragraphs'][0])}")
print(f"Keys of cuad_data[0]['paragraphs'][0]: {cuad_data[0]["paragraphs"][0].keys()}")

Type of cuad_data[0]['paragraphs']: <class 'list'>
Length of cuad_data[0]['paragraphs]: 1

Type of cuad_data[0]['paragraphs'][0]: <class 'dict'>
Length of cuad_data[0]['paragraphs'][0]: 2
Keys of cuad_data[0]['paragraphs'][0]: dict_keys(['qas', 'context'])


1.   You are accessing each contract's paragraphs, which is a **list**.
2.   Each list of paragraphs contains a dictionary, which has two keys: **qas** and **context**



In [8]:
print(f"Type of cuad_data[0]['paragraphs'][0]['qas']: {type(cuad_data[0]['paragraphs'][0]['qas'])}")
print(f"Length of cuad_data[0]['paragraphs'][0]['qas']: {len((cuad_data[0]['paragraphs'][0]['qas']))}")

print("\nLooking at a single qas:")
print(cuad_data[0]['paragraphs'][0]['qas'][0])
print(f"Keys in a single qas: {cuad_data[0]['paragraphs'][0]['qas'][0].keys()}")

Type of cuad_data[0]['paragraphs'][0]['qas']: <class 'list'>
Length of cuad_data[0]['paragraphs'][0]['qas']: 41

Looking at a single qas:
{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}
Keys in a single qas: dict_keys(['answers', 'id', 'question', 'is_impossible'])


In [9]:
cuad_data[0]['paragraphs'][0]['qas'][4]["answers"][0]

{'text': 'The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.',
 'answer_start': 5268}

1.   **qas** is a list. Each item in the list is a dictionary.
2.   A single item in the **qas** list has the following keys: **answers**, **id**, **question**, and **is_impossible**.



In [10]:
cuad_data[0]['paragraphs'][0].keys()

dict_keys(['qas', 'context'])

In [11]:
# inspect the first contract's paragraph
cuad_data[0]["paragraphs"][0].keys() # each paragraph contains 'qas' and 'context'

# inspect the qas first; inspect the first item in the list
cuad_data[0]["paragraphs"][0]["qas"][0] # each qas item contains 'answers' which is a list of its own, 'id', 'question', and 'is_impossible'

# focussing on a single paragraph
single_paragraph = cuad_data[0]["paragraphs"][0]
# focussing on the same anchor text
anchor_text = single_paragraph["context"]

# loop through all questions inside the single paragraph
for qa in single_paragraph["qas"]:
  # skip categories where no terms were found
  if qa["is_impossible"]:
    continue

  clause_id = qa["id"].split("__")[-1]
  question = qa["question"]
  answers = [ans["text"] for ans in qa["answers"]]

  # print out the findings
  print(f"ID: {clause_id}")
  print(f"QUESTION: {question}")
  print(f"ANSWER: {answers}")
  print("="*40)

ID: Document Name
QUESTION: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
ANSWER: ['DISTRIBUTOR AGREEMENT']
ID: Parties
QUESTION: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
ANSWER: ['Distributor', 'Electric City Corp.', 'Electric City of Illinois L.L.C.', 'Company', 'Electric City of Illinois LLC']
ID: Agreement Date
QUESTION: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
ANSWER: ['7th day of September, 1999.']
ID: Effective Date
QUESTION: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective 
ANSWER: ['The term of this  Agreement  shall be ten (10)                        

In [12]:
anchor_text

'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  hav

In [13]:
# extract the unique clauses from the contracts
unique_clauses = set()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]
      unique_clauses.add(clause_id)

print("Unique clauses found in the first 5 contracts:")
print(unique_clauses)
print(f"Number of unique clauses found: {len(unique_clauses)}")

Unique clauses found in the first 5 contracts:
{'Audit Rights', 'Joint Ip Ownership', 'Rofr/Rofo/Rofn', 'Revenue/Profit Sharing', 'Renewal Term', 'Competitive Restriction Exception', 'License Grant', 'Change Of Control', 'Exclusivity', 'Non-Transferable License', 'Third Party Beneficiary', 'Price Restrictions', 'Effective Date', 'Irrevocable Or Perpetual License', 'Governing Law', 'Termination For Convenience', 'Expiration Date', 'Source Code Escrow', 'Uncapped Liability', 'Cap On Liability', 'Non-Compete', 'Covenant Not To Sue', 'No-Solicit Of Employees', 'Minimum Commitment', 'No-Solicit Of Customers', 'Affiliate License-Licensee', 'Non-Disparagement', 'Notice Period To Terminate Renewal', 'Document Name', 'Warranty Duration', 'Agreement Date', 'Ip Ownership Assignment', 'Unlimited/All-You-Can-Eat-License', 'Affiliate License-Licensor', 'Parties', 'Volume Restriction', 'Post-Termination Services', 'Insurance', 'Liquidated Damages', 'Most Favored Nation', 'Anti-Assignment'}
Number of 

The following tiers will be used for filtering the nature of clauses in order of how rigorously they can be litigated in courts, or be subject to intense pragmatic inference. 

* TIER 1 and TIER 2 contain the natures that are most sort for in terms of how they can be litigated in courts
* TIER 3 is entirely deterministic with clearly outlined guardrails, constraints, numerical and quantitative boundaries, and thresholds that cannot be subjectively reasoned over

The main focus is on TIER 1 and 2

In [14]:
# HIGH PRIORITY CLAUSES THAT ARE SUBJECT TO INTENSE PRAGMATIC INFERENCE
# AND LITIGATION IN COURTS
tier_1_clauses = {
    'Audit Rights',
    'Termination For Convenience',
    'Most Favored Nation',
    'Non-Compete',
    'Insurance',
    'Minimum Commitment',
    'Post-Termination Services',
    'Warranty Duration'
}

# STRICT PROHIBITIONS BUT CAN ALSO BE LITIGATED IN COURTS BASED ON THE
# CONTEXT OF THE CONTRACT AND CASE
tier_2_clauses = {
    'Exclusivity',
    'Anti-Assignment',
    'Cap On Liability',
    'Competitive Restriction Exception',
    'Covenant Not To Sue',
    'No-Solicit Of Customers',
    'No-Solicit Of Employees',
    'Non-Disparagement',
    'Non-Transferable License',
    'Revenue/Profit Sharing',
    'Rofr/Rofo/Rofn',
    'Source Code Escrow',
    'Third Party Beneficiary',
    'Price Restrictions',
    'License Grant',
    'Affiliate License-Licensor',
    'Affiliate License-Licensee',
    'Ip Ownership Assignment',
    'Joint Ip Ownership',
    'Irrevocable Or Perpetual License',
    'Unlimited/All-You-Can-Eat-License',
    'Volume Restriction'
}

# DETERMINISTIC OPERATIONS AND CLAUSES; DATES, AMOUNT, NUMERICAL VALUES, AND
# OTHER CLEAR DETERMINISTIC OPERATIONS
tier_3_clauses = {
    'Agreement Date',
    'Document Name',
    'Effective Date',
    'Expiration Date',
    'Governing Law',
    'Liquidated Damages',
    'Notice Period To Terminate Renewal',
    'Renewal Term',
    'Parties',
    'Uncapped Liability'
}

The extended `vagueness_seed_set` contains the vague terms along with their different synonym variations and paraphrase-like words needed for filtering, and extracting the vague terms from the actual clause text.

In [15]:
vagueness_seed_set = {
    # ── Effort & diligence ──────────────────────────────────────────────
    "reasonable efforts", "best efforts", "commercially reasonable",
    "due diligence", "reasonable care", "good faith", "workmanlike manner",
    "best practices", "reasonable endeavours", "all reasonable steps",
    "every reasonable effort", "diligent efforts", "reasonable commercial efforts",
    "utmost care", "exercise of judgment", "commercially practicable",
    "economically reasonable", "technically feasible",
    "consistent with good industry practice", "as would a prudent operator",
    "acting reasonably", "using its discretion",

    # ── Time & urgency ──────────────────────────────────────────────────
    "promptly", "in a timely manner", "as soon as practicable",
    "without undue delay", "for a reasonable period",
    "termination of this Agreement", "from time to time", "periodic",
    "duration", "seasonable", "business hours", "within a reasonable time",
    "with all due speed", "expeditiously", "at the earliest opportunity",
    "without unnecessary delay", "within a commercially reasonable period",
    "in due course", "forthwith", "in due time", "on a timely basis",
    "in the near term", "shortly after", "when practicable",
    "upon reasonable notice", "reasonable notice period",

    # ── Scope, degree & quantity ────────────────────────────────────────
    "material", "substantial", "limited", "relevant", "related", "generally",
    "appropriate", "similar", "de minimis", "significant", "incidental",
    "including but not limited to", "inter alia", "and/or",
    "save as otherwise provided", "appreciable", "meaningful", "non-trivial",
    "measurable", "proportionate", "commensurate", "reasonably proportionate",
    "unduly burdensome", "reasonably necessary", "to the extent practicable",
    "to a reasonable extent", "without limitation", "as applicable",
    "where relevant", "as appropriate", "to the extent required",

    # ── Harm, change & threshold ────────────────────────────────────────
    "material adverse effect", "material breach", "material adverse change",
    "material adverse impact", "material adverse consequence",
    "materially and adversely", "substantial impairment", "material disruption",
    "material deviation", "materially prejudice", "disproportionate impact",
    "unreasonable hardship", "undue prejudice", "undue harm", "undue risk",

    # ── Necessity & discretion ──────────────────────────────────────────
    "necessary", "sole discretion", "need to know", "confidential nature",
    "adequate", "satisfactory", "proper", "intended purpose",
    "not to be unreasonably withheld", "mutual satisfaction", "at its option",
    "consultation", "absolute discretion", "unfettered discretion",
    "not to be unreasonably delayed", "not to be unreasonably conditioned",
    "without arbitrary restriction", "reasonably required",
    "reasonably requested", "if deemed appropriate", "as deemed necessary",
    "in its reasonable opinion", "acting in good faith",
    "in its reasonable judgment", "as it sees fit", "as directed",

    # ── Industry norms & quality ────────────────────────────────────────
    "customary", "ordinary course of business", "industry standard",
    "standard practice", "normally", "comparable", "acceptable",
    "conventional", "fit for purpose", "first-class condition",
    "commercially sensitive", "prevailing market practice",
    "generally accepted practice", "market standard",
    "accepted industry norms", "standard market terms",
    "customary market conditions", "in accordance with accepted methods",
    "consistent with past practice", "as is customary",
    "in accordance with best available techniques",
    "reasonable engineering standards", "professionally acceptable",
    "to a professional standard", "of merchantable quality",
    "of satisfactory quality",

    # ── Knowledge, intent & foresight ──────────────────────────────────
    "foreseeable", "contemplated", "intended", "anticipated", "applicable",
    "knowledge", "directly or indirectly", "disclosed in confidence",
    "all copies", "survive", "mutual agreement", "substantially similar",
    "reasonable expectations", "actual knowledge", "constructive knowledge",
    "reasonably should have known", "to the best of its knowledge",
    "as far as it is aware", "reasonably foreseeable",
    "unforeseen circumstances", "unanticipated events",
    "beyond reasonable expectation", "reasonable belief", "bona fide belief",
    "reasonable grounds", "having regard to all circumstances",

     # ── Confidentiality & information ───────────────────────────────────
    "proprietary information", "non-public information",
    "sensitive business information", "trade secrets",
    "sufficiently confidential", "reasonably considered confidential",
    "maintained in confidence", "treated as confidential",
    "in accordance with confidentiality obligations",

    # ── Financial & commercial terms ────────────────────────────────────
    "commercially attractive", "economically viable",
    "commercially justifiable", "at a reasonable price", "fair market value",
    "arm's length", "at prevailing rates", "on reasonable commercial terms",
    "on competitive terms", "at a rate reflecting market conditions",
    "at cost", "without unreasonable mark-up", "reasonable compensation",
    "reasonable fees",

    # ── Survival, agreement & modification ─────────────────────────────
    "notwithstanding the foregoing", "without prejudice to",
    "subject to the foregoing", "except as otherwise agreed",
    "unless otherwise specified", "where not inconsistent", "insofar as",
    "to the fullest extent permitted by law", "as may be amended",
    "as modified from time to time", "by mutual written consent",
}

len(vagueness_seed_set)

206

### Extracting clause categories and texts

Using the dataset available, the following needs to be extracted:

*   Document ID (for cross referencing later if needed)
*   Clause category/ID
*   Text from the clause that describes the contract
*   The tier it belongs to
*   The set of vague terms contained in it

Furthermore, we also need to check whether the clause is:

*   Lethal - belongs to tier 1
*   Has context risk - belongs to tier 2

All of these will be stored as a dictionary, and housed in a list



In [16]:
clause_and_terms = list()

for contract in cuad_data:
  # access the paragraphs in each contract
  for paragraph in contract["paragraphs"]:
    # access the qas of each paragraph
    qas = paragraph["qas"]
    # extract the ids from each qa
    for qa in qas:
      # check_lethality flags if the clause belongs to tier 1
      # context_risk flags if the clause belongs to tier 2
      check_lethality, context_risk = False, False
      tier = 0

      if qa["is_impossible"]:
        continue
      clause_id = qa["id"].split("__")[-1]

      # check if the clause id belongs to tier 3, if yes, discard it
      if clause_id in tier_3_clauses:
        continue

      # check tier and assign label
      if clause_id in tier_1_clauses:
        tier = 1
      elif clause_id in tier_2_clauses:
        tier = 2

      clause_text = qa["answers"][0]["text"]
      contract_id = contract['title']
      vague_terms = {term for term in vagueness_seed_set if term in clause_text}


      # check the lethality of the clause
      if tier == 1 and vague_terms:
        check_lethality = True
        context_risk = False
      elif tier == 2 and vague_terms:
        check_lethality = False
        context_risk = True

      if check_lethality or context_risk:
        clause_term_dict = {
            "contract_id": contract_id,
            "clause_category": clause_id,
            "clause_text": clause_text,
            "tier": tier,
            "vague_terms": vague_terms,
            "is_lethal": check_lethality,
            "has_context_risk": context_risk
        }

        clause_and_terms.append(clause_term_dict)

In [17]:
print(f"Extracted data: {len(clause_and_terms)}")

Extracted data: 1428


In [18]:
from prompt_toolkit.shortcuts import print_container
import random

sampling_size = int(len(clause_and_terms) * 0.20)
compressed_sampling_size = int(sampling_size * 0.05)

for i in range(compressed_sampling_size):
  random_idx = random.randint(0, len(clause_and_terms)-1)

  print(f"CLAUSE CATEGORY: {clause_and_terms[random_idx]["clause_category"]}")
  print(f"CLAUSE TEXT: {clause_and_terms[random_idx]["clause_text"]}")
  print(f"TIER: {clause_and_terms[random_idx]["tier"]}")
  print(f"VAGUE TERMS: {clause_and_terms[random_idx]["vague_terms"]}")
  print(f"IS LETHAL: {clause_and_terms[random_idx]["is_lethal"]}")
  print(f"HAS CONTEXT RISK: {clause_and_terms[random_idx]["has_context_risk"]}")
  print("="*40, end="\n\n")

CLAUSE CATEGORY: Ip Ownership Assignment
CLAUSE TEXT: In such case: (i) TPH-A or TPH, as the case may be, shall acquire sole and exclusive title to the GaN Equipment, free and clear of all Encumbrances, and none of FSL, AFSL or the Company shall have any right, title or interest in such GaN Equipment, (ii) such GaN Equipment shall be clearly labeled as the property of TPH-A or TPH, as the case may be, and (iii) FSL and AFSL shall cause to be assigned to TPH-A or TPH, as the case may be, all licenses and warranties for such GaN Equipment and the software or firmware required to operate such GaN Equipment that are attached to, installed on, or embodied in such GaN Equipment as of the Effective Date.
TIER: 2
VAGUE TERMS: {'proper'}
IS LETHAL: False
HAS CONTEXT RISK: True

CLAUSE CATEGORY: Non-Compete
CLAUSE TEXT: During the Term, except as otherwise provided in this Agreement, Network Affiliate and its affiliates agree not to engage or  participate in any business, hold equity interests, 

Right now, each data point `clause_and_terms` houses the vague clauses, vague terms, and the accompanying signals such as lethality and context risks.

From this, we move on to semantic mapping with the case law from the Harvard Corpus, i.e., [COLD Cases](https://huggingface.co/datasets/harvard-lil/cold-cases).

The `clause_text` present in each data point will be used to semantically map the right case law from this corpus, which will be the judicial prose of the triplet.



## Semantic Mapping with Harvard LIL COLD Cases

Taking `clause_and_terms`, we will use the `clause_text` key/item and encode it into its **vector embeddings**.

A judge will not always write the contractual clauses as and how it appears in a contract in his reasoning; hence, string matching will fail. We need to map them in a dense **vector space**.

In [49]:
from sentence_transformers import SentenceTransformer
import numpy as np

# intialise the asymmetric Gemma embedding model
bi_encoder = SentenceTransformer("google/embeddinggemma-300m")

# # cehck if kaggle successfully mounted both the T4 accelerators
if torch.cuda.device_count() > 1:
    print(f"Accelerating the pipeline across {torch.cuda.device_count()} T4 GPUs")

    # create a pool across all available cuda devices
    cuda_pool = bi_encoder.start_multi_process_pool(
        target_devices=["cuda:0", "cuda:1"]
    )
else:
    cuda_pool = None

# # # wrap the model layers to split the encoding batches automatically across both the cores
# bi_encoder._first_module().auto_model = torch.nn.DataParallel(bi_encoder._first_module().auto_model)
# bi_encoder._first_module().auto_model.config = bi_encoder._first_module().auto_model.module.config

# # cast the unified graph directly to the primary cuda stream
# bi_encoder = bi_encoder.to("cuda")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Accelerating the pipeline across 2 T4 GPUs


In [60]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')
cross_encoder.model.to("cuda")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12, e

In [20]:
cuda_pool

{'input': <multiprocessing.queues.Queue at 0x7b22650c2c30>,
 'output': <multiprocessing.queues.Queue at 0x7b223e95c2f0>,
 'processes': [<SpawnProcess name='SpawnProcess-1' pid=1012 parent=955 started daemon>,
  <SpawnProcess name='SpawnProcess-2' pid=1026 parent=955 started daemon>]}

In [21]:
# vectorize the clause_texts in each data point in clause_and_terms
anchor_clause_texts = [data_point["clause_text"] for data_point in clause_and_terms]

# Use the standard .encode() method instead
# CUAD anchors — these are the queries
anchor_clause_embeddings_np = bi_encoder.encode(
    anchor_clause_texts,
    pool=cuda_pool,
    prompt="query: ", # query side prompt
    show_progress_bar=True,
    batch_size=32,
    convert_to_tensor=False
)

anchor_clause_embeddings = torch.from_numpy(anchor_clause_embeddings_np).to("cuda")

# # Appellate windows — these are the documents
# window_embeddings_np = bi_encoder.encode(
#     text_blocks,
#     pool=cuda_pool,
#     prompt="passage: ",
#     show_progress_bar=True, # document size prompt
#     batch_size=32,
#     convert_to_tensor=False
# )

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

## Dual-Vector Anchoring

### Implementing a coarse filter on harvard-lil/cold-cases

The coarse filter is not looking for contratual obligations here. Rather, it's focus is to filter out all sentences from the opinion texts that are not useful for the vector embeddings to compute similarity with.

From a micro perspective, the filter will:

*   Check if the opinion texts belong to **contract, commercial law, corporate law, or other such similar cases.** No statutory, administrative, or tort law cases will be considered.
*   Segment each opinion text into individual sentences. Use sentences that have $> 10$ and $\le 100$ words in a sentence. This can be tuned depending on the quality of the filter's output.
*   Ensure **no statutory sentences** are present. *This is important.*
*   No checking for modals or seed words (vague terms) in this step.





In [22]:
# Statutory signals
STATUTORY_PATTERN = re.compile(
    r'\b(§|Code|Section|Article|Constitution|Statute|Act of \d{4}|'
    r'U\.S\.C\.|W\.Va\. Code|provided by law|mandates that|'
    r'every municipality|any person|no employee|all employers)\b',
    re.IGNORECASE
)

In [23]:
# structural indicators for retrospective reasoning prose
REASONING_INDICATORS = re.compile(
  # Epistemic / judgment verbs (strong signals)
  r'\b(held|concluded|found|determined|argued|asserted|interpreted|appealed|testified)\b|'
  r'\b(finds|found|ruled|decided|opined|considered|recognised|acknowledged)\b|'
  r'\b(construed|construction|construing)\b|'

  # First-person judicial voice (strong signals)
  r'\b(I\s+find|I\s+am\s+satisfied|I\s+consider|I\s+prefer|I\s+accept|I\s+reject)\b|'
  r'\b(I\s+agree|I\s+disagree|I\s+conclude|I\s+hold|in\s+my\s+judgment|in\s+my\s+view)\b|'
  r'\b(the\s+court\s+finds|the\s+court\s+held|the\s+court\s+concludes|the\s+court\s+considers)\b|'

  # Plain meaning / interpretive analysis (strong signals)
  r'\b(on\s+a\s+proper\s+construction|the\s+plain\s+meaning|the\s+natural\s+meaning)\b|'
  r'\b(purposive|contextual\s+reading|read\s+as\s+a\s+whole|read\s+together)\b|'
  r'\b(the\s+better\s+view|the\s+correct\s+interpretation|properly\s+construed)\b|'

  # Inferential / consequential connectors (medium signals)
  r'\b(therefore|accordingly|it\s+follows|consequently|necessarily\s+means)\b|'
  r'\b(it\s+is\s+clear\s+that|it\s+must\s+follow|this\s+suggests|this\s+indicates)\b|'
  r'\b(would\s+have|could\s+have|should\s+have|must\s+have)\b|'

  # Reasonableness / obligation framing (medium signals)
  r'\b(reasonable|reasonably|unreasonable|unreasonably)\b|'
  r'\b(the\s+parties\s+intended|the\s+intention\s+of\s+the\s+parties|objectively\s+construed)\b|'
  r'\b(implied\s+term|implied\s+obligation|necessary\s+implication)\b|'

  # Comparative / distinguishing reasoning (medium signals)
  r'\b(unlike|by\s+contrast|whereas|distinguished\s+from|analogous\s+to)\b|'
  r'\b(consistent\s+with|inconsistent\s+with|contrary\s+to)\b|'

  # Original weak signals retained
  r'\b(this\s+provision|such\s+obligation|the\s+disputed|underlying\s+contract)\b',

  re.IGNORECASE
)

In [24]:
def filter_cold_cases_opinion_text(
    opinion_text: str,
    window_size: int = 3,
    step_size: int = 1
    ):
  """
  """

  # store all valid sentences individually, not as an entire opinion text
  clause_sentences = []
  reasoning_sentences = []

  # split the entire opinion text into individual sentences
  sentences = re.split(r'(?<=[.!?])\s+', opinion_text)

  for sentence in sentences:
    sentence = sentence.strip() # strip off all leading and trailing whitespaces

    # check for statutory patterns and skip them
    if STATUTORY_PATTERN.search(sentence):
      continue

    # check the number of words in the sentence and eliminate accordingly
    words = sentence.split()
    if len(words) < 10 or len(words) > 100:
      continue

    # the gatekeeper routing to embedding or RRL
    if REASONING_INDICATORS.search(sentence):
      reasoning_sentences.append(sentence)
    else:
      clause_sentences.append(sentence)

  # check if valid sentences is not empty, and construct the sliding windows
  if clause_sentences:

    # construct the overlapping sliding windows for the clause track
    sliding_windows = []
    num_sentences = len(clause_sentences)

    if num_sentences >= window_size:
      # loop from the starting of the sentences list
      # up until the last window_size element+1
      # step the control by step size
      # this creates the sliding window
      for i in range(0, num_sentences - window_size + 1, step_size):
        # join window_size sentences into one text block
        window_text = " ".join(clause_sentences[i: i + window_size])
        sliding_windows.append({
            "text_window": window_text,
            "start_idx": i,
            "end_idx": i + window_size
        })

    # valid_sentences_embeddings = embedding_model.encode(
    #     clause_sentences,
    #     show_progress_bar=False,
    #     batch_size=64
    # )

    # check the cosine similarity with the cuad anchors
    # similarities = embedding_model.similarity(
    #     anchor_clause_embeddings,
    #     valid_sentences_embeddings
    # )

    output = {
        "opinion_text": opinion_text,
        "sentences": clause_sentences,
        "clause_sliding_windows": sliding_windows,
        "reasoning_sentences": reasoning_sentences
    }

    return output

  else:
    return False

In [25]:
from datasets import load_dataset

# load the dataset from HuggingFace: https://huggingface.co/datasets/harvard-lil/cold-cases
cold_cases = load_dataset(
    "harvard-lil/cold-cases",
    split="train",
    streaming=True
)

cold_cases["opinions"]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

In [26]:
# parse the dataset and check for all the different case natures
case_natures = set()

# get all the different natures of cases
for idx, case in enumerate(cold_cases):
  if idx >= 1000000:
    break
  nature = case.get("nature_of_suit", "") or "" # read the nature of the case
  case_natures.add(nature)

len(case_natures), case_natures

Process SpawnProcess-1:
Process SpawnProcess-2:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py", line 1543, in _encode_multi_process_worker
    chunk_id, inputs, kwargs = input_queue.get()
                               ^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 102, in get
    with self._rlock:
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing

KeyboardInterrupt: 

In [27]:
target_case_natures = {
    # Original set
    'Tort, Contract, and Real Property',
    'Private Civil Diversity',
    'Private Civil Federal',
    'OIL, GAS AND MINERALS',
    # New additions
    'CONTRACTS',
    'DEBTOR/CREDITOR',
    'EMPLOYER/EMPLOYEE DISPUTE',
    'FORECLOSURE',
    'LANDLORD/TENANT',
    'REAL PROPERTY',
    'PROBATE - WILLS - TRUSTS',
    'United States Civil',
    'Civil',
    'civil',
    'Bankruptcy',
    'Bankruptcy Direct from BC',
    'Bankruptcy From District Court',
    'bankruptcy from bankruptcy court',
    'bankruptcy from district court',
    'Revenue (Tax)',
    'WORKERS COMPENSATION',
}

In [28]:
# count = 0

# for case in cold_cases:
#   if count >= 10:
#     break

#   nature = case.get("nature_of_suit", "") or "" # read the nature of the case
#   # only proceed with the targetted case natures
#   if nature in target_case_natures:

#       opinions = case.get("opinions", []) # extract the opinions into a list

#       # check if the case has an opinion text that can be extracted
#       if opinions and opinions[0].get("opinion_text"):
#         validity_status = filter_cold_cases_opinion_text(opinions[0]['opinion_text'])

#         # if the filter returns false, skip document
#         if not validity_status:
#           continue

#         # here now, we need to isolate the high scoring coordinates
#         # if the cosine similarity >= 70
#         # pair the raw case sentence text directly with the metadata of the
#         # matching CUAD anchor clause
#         row_indices, col_indices = torch.where(validity_status['similarities'] >= 0.70)

#         for row, col in zip(row_indices, col_indices):
#           row_idx = row.item()
#           col_idx = col.item()

#           # extract the matching score from the matrix
#           score = validity_status['similarities'][row_idx][col_idx].item()

#           # obtain the matching cuad metadata from the json directly
#           # row_idx looks up the cuad anchors
#           matching_cuad_metadata = clause_and_terms[row_idx]

#           # col_idx looks up the raw sentence from the valid sentences
#           matching_raw_sentence = validity_status["sentences"][col_idx]

#           print(f"Matching Score: {score: .2f}")
#           print(f"|-- CUAD CATEGORY: {matching_cuad_metadata["clause_category"]}")
#           print(f"|-- CUAD Anchor: {matching_cuad_metadata["clause_text"]}")
#           print(f"└── Raw sentence: {matching_raw_sentence}")
#           print("="*50)

#           count += 1

**Notes until this point in the extraction:**

1.  The threshold bucket that contains scores higher than 75% contains pure lexical matchings which are similar to the substring matching that can be achieved using pure code
2.  The bucket with scorings in the 70-75% range house a lot of text is judicial reasoning, rather than the clauses.
    *   The sentences assertive, stating the judge's interpretation of the clauses and its obligations, **not** the clause itself
    *   These are mappings that are needed for the interpretation part of the triplet structure which will what the reward system will rely on
    *   Failure to isolate these two aspects will corrupt the learning process of the reinforcement learning loop
3.  Surprisingly, the 60-70% bucket contains positive signals where it had identified clauses, some of then including:

    ```
    |-- CUAD CATEGORY: Termination For Convenience
    |-- CUAD Anchor: Either party may, at its option, terminate this Agreement       without cause, effective at any time after January 31, 1999, upon giving       at least ninety (90) days prior written notice of such termination to the       other party.
    └── Raw sentence: Rather, the Advisory Agreements provided for termination without cause upon sixty days’ written notice.
    ```

    This shows promising results. It also contains a lot of structural noise compared to the other two buckets.

## Reverse Constructing the JDAR Triplet

Given that we already have the anchor clauses, i.e., the anchor clause, clause text, and the vague terms that were extracted from CUAD. Using these clause texts and the judical reasoning candidates that we have, we can use vector spaces to pair the clause texts with their judicial interpretation counterparts.

In essence, the triplet has the following structure:

$
\text{JDAR Triplet Data Point} = \begin{cases}
\textbf{Clause} : & \text{Clean CUAD Text Snippet} \\
\textbf{Terms} : & \text{Extracted Seed Words from the CUAD snippet and the judicial interpretation} \\
\textbf{Reasoning} : & \text{Aligned Harvard-LIL COLD Cases Opinion Sentence (i.e., the extracted reasoning sentences}
\end{cases}
$

Right now, we have a lot more judicial reasoning sentences that try to interpret the clauses than the actual clauses themselves. If we were to create a dataset using this, we would end up with a highly **imbalanced dataset.** However, there is something else that we can do, one that can highly speed up this process of constructing the triplet structure.

In [33]:
STRUCTURAL_ANCHORS = re.compile(
    r'\b(agreement|section|article|clause|provision|policy|schedule|exhibit|hereunder|parties)\b'
    r'|'
    r'\b(constru|interpret|harmonize|give effect to|render.{1,20}surplusage|in interpreting|'
    r'the language of|the term|parties intended|must be read|read in|no work to do|'
    r'ascertainable|economic benefit|notice.{1,20}requirement|the contract.{1,20}requires)\b',
    re.IGNORECASE
)

In [34]:
STRUCTURAL_ANCHOR_WORDS = re.compile(
    r'\b(agreement|section|article|clause|provision|policy|schedule|exhibit|hereunder|parties)\b',
    re.IGNORECASE
)

INTERPRETIVE_SIGNALS = re.compile(
    r'\b(constru|interpret|harmonize|give effect|surplusage|the term|intended|must be read|'
    r'in interpreting|the language|renders?\b.{1,20}\bmeaningless|no work to do|'
    r'ascertainable|economic benefit|contractual obligation|the provision requires|'
    r'we read|we hold|the contract requires|ambiguous|vague|definite)\b',
    re.IGNORECASE
)

In [36]:
# Target the non-commercial domains dominating your false positives
DOMAIN_EXCLUSION = re.compile(
    r'\b(medical|malpractice|panel|doctor|patient|decedent|spouse|widow|estate|elective\s+share|'
    r'divorce|testimony|criminal|guilty|'
    # Add these
    r'fictitious|never served|has not appeared|has neither been served|'
    r'28\s+u\.s\.c|§\s*1291|footnote|fn\.\s*\d|see note \d|'
    r'plaintiff also requested|were denied|for failure to comply|'
    r'successor by merger|national association)\b',
    re.IGNORECASE
)

In [37]:
# PHASE 1: CPU Preprocessing — accumulate all windows

all_text_blocks = []        # flat list of all window texts
case_metadata_store = {}    # dict, not list

# the global window will be used to track the overlap of windows and thereby prevent 
# such overlapping
global_window_id = 0

for case_idx, case in enumerate(tqdm(cold_cases, desc="Preprocessing")):
    nature = case.get("nature_of_suit", "") or ""
    if nature not in target_case_natures:
        continue

    opinions = case.get("opinions", [])
    if not (opinions and opinions[0].get("opinion_text")):
        continue

    opinion_text = opinions[0]["opinion_text"]

    # keep your existing 150k characters guardrail to skip immense opinion texts
    if len(opinion_text) > 150000:
        continue

    # filter out the necessary cases and obtain the output. The output will have the following structure
    #     output = {
    #        "opinion_text": opinion_text,
    #        "sentences": clause_sentences,
    #        "clause_sliding_windows": sliding_windows,
    #        "reasoning_sentences": reasoning_sentences
    #    }
    
    # case_data = filter_cold_cases_opinion_text(
    #     opinion_text=opinion_text,
    #     window_size=3,
    #     step_size=1
    # )
    case_data = filter_cold_cases_opinion_text(
        opinion_text, 
        window_size=3, 
        step_size=2
    )

    if not case_data or not case_data["clause_sliding_windows"]:
        continue

    # pre-filter windows using the existing regex gates
    filtered_texts = []
    filtered_windows = []

    for window in case_data["clause_sliding_windows"]:
        window_text = window["text_window"]
        if DOMAIN_EXCLUSION.search(window_text):
            continue
        # AFTER — requires structural word AND interpretive signal
        if not STRUCTURAL_ANCHOR_WORDS.search(window_text):
            continue
        if not INTERPRETIVE_SIGNALS.search(window_text):
            continue
        filtered_texts.append(window_text)
        filtered_windows.append(window)

    if not filtered_texts:
        continue

    start_id = global_window_id
    all_text_blocks.extend(filtered_texts)
    global_window_id += len(filtered_texts)  # ← advance by actual count

    case_metadata_store[case_idx] = {
        "case": case,
        "case_data": case_data,
        "filtered_windows": filtered_windows,
        "filtered_texts": filtered_texts,
        "window_slice": (start_id, global_window_id),
    }

print(f"Accumulated {len(all_text_blocks)} windows from {len(case_metadata_store)} cases")


# # PHASE 2: GPU mega-batch encoding — one call, both GPUs loaded

# all_embeddings_np = bi_encoder.encode(
#     all_text_blocks,
#     pool=cuda_pool,
#     batch_size=256,
#     prompt="passage: ",
#     show_progress_bar=True,
#     convert_to_tensor=False,
# )

# all_embeddings = torch.from_numpy(all_embeddings_np).to("cuda")

Preprocessing: 0it [00:00, ?it/s]

Accumulated 16040 windows from 6873 cases


In [44]:
import gc

gc.collect()
torch.cuda.empty_cache()

# if you have two GPUs, clear both explicitly
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()

In [50]:
# PHASE 2: GPU mega-batch encoding — one call, both GPUs loaded

all_embeddings_np = bi_encoder.encode(
    all_text_blocks,
    pool=cuda_pool,
    batch_size=64,
    prompt="passage: ",
    show_progress_bar=True,
    convert_to_tensor=False
)

all_embeddings = torch.from_numpy(all_embeddings_np).to("cuda")

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

In [61]:
# PHASE 3: Vectorized similarity + result collection
threshold = 0.40
cross_threshold = -6.0  # permissive floor — eliminates only truly irrelevant pairs

distributed_threshold_results = {
    threshold: [],
    "reasoning_sentences": [],
}

# Stage 1 — bi-encoder coarse similarity (UNCHANGED)
all_similarities = bi_encoder.similarity(
    anchor_clause_embeddings,
    all_embeddings
)

matches_pbar = tqdm(total=None, desc="Matches Found", unit="match", position=0, leave=True)

for case_idx, meta in tqdm(case_metadata_store.items(), desc="Scoring Cases", position=1, leave=True):
    start, end = meta["window_slice"]
    case_sims = all_similarities[:, start:end]

    if meta["case_data"].get("reasoning_sentences"):
        distributed_threshold_results["reasoning_sentences"].extend(
            meta["case_data"]["reasoning_sentences"]
        )

    threshold_mask = case_sims >= threshold
    row_indices, col_indices = torch.where(threshold_mask)

    if len(row_indices) == 0:
        continue

    # ── cross-category confirmation map ──
    window_to_categories_map = {}
    for row, col in zip(row_indices, col_indices):
        r_idx = row.item()
        c_idx = col.item()
        category = clause_and_terms[r_idx]["clause_category"]
        if c_idx not in window_to_categories_map:
            window_to_categories_map[c_idx] = set()
        window_to_categories_map[c_idx].add(category)

    # ── collect candidates that pass all gates ──
    used_sentence_indices = set()
    candidates = []

    for row, col in zip(row_indices, col_indices):
        row_idx = row.item()
        col_idx = col.item()
        score = case_sims[row_idx, col_idx].item()

        if score >= threshold:
            if len(window_to_categories_map[col_idx]) > 2:
                continue

            matching_cuad_metadata = clause_and_terms[row_idx]
            matching_raw_window_dict = meta["filtered_windows"][col_idx]
            matching_raw_window = meta["filtered_texts"][col_idx]

            start_idx = matching_raw_window_dict.get(
                "start_idx", matching_raw_window_dict.get("start_sentence_idx")
            )
            end_idx = matching_raw_window_dict.get(
                "end_idx", matching_raw_window_dict.get("end_sentence_idx")
            )
            window_range = set(range(start_idx, end_idx + 1))

            if len(window_range.intersection(used_sentence_indices)) > 1:
                continue

            used_sentence_indices.update(window_range)

            candidates.append({
                "cuad_category": matching_cuad_metadata["clause_category"],
                "cuad_anchor": matching_cuad_metadata["clause_text"],
                "raw_sentence": matching_raw_window,
                "bi_encoder_score": score,
                "contract_id": meta["case"].get("title", "Unknown_Contract"),
            })

    # ── Stage 2: cross-encoder re-ranking ──
    if candidates:
        ce_pairs = [
            [c["cuad_anchor"], c["raw_sentence"]]
            for c in candidates
        ]
        cross_scores = cross_encoder.predict(ce_pairs, batch_size=64)

        for candidate, cross_score in zip(candidates, cross_scores):
            if cross_score > cross_threshold:
                candidate["cross_encoder_score"] = float(cross_score)
                distributed_threshold_results[threshold].append(candidate)
                matches_pbar.update(1)

matches_pbar.close()
print(f"\nFull Run Complete. Found {len(distributed_threshold_results[threshold])} matches across {len(case_metadata_store)} qualifying cases.")

Matches Found: 0match [00:00, ?match/s]

Scoring Cases:   0%|          | 0/6873 [00:00<?, ?it/s]


Full Run Complete. Found 1005 matches across 6873 qualifying cases.


In [62]:
import pandas as pd

# (columnar, compressed, analysis-ready)
matches_df = pd.DataFrame(distributed_threshold_results[threshold])
matches_df.to_parquet("clause_matches_version3.parquet", engine="pyarrow", index=False)

# Reasoning sentences → Parquet (single column)
reasoning_df = pd.DataFrame({
    "reasoning_sentence": distributed_threshold_results["reasoning_sentences"]
})
reasoning_df.to_parquet("reasoning_sentences_version3.parquet", engine="pyarrow", index=False)

# Full dict → JSON (complete snapshot)
json_safe = {
    str(threshold): distributed_threshold_results[threshold],
    "reasoning_sentences": distributed_threshold_results["reasoning_sentences"],
}

with open("distributed_threshold_results_version3.json", "w") as f:
    json.dump(json_safe, f, indent=2)

print(f"Saved {len(matches_df)} matches + {len(reasoning_df)} reasoning sentences")

Saved 1005 matches + 220831 reasoning sentences


In [ ]:
# output_path = 'extracted_triplets_full.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(distributed_threshold_results[0.40], f, indent=4, ensure_ascii=False)

# print(f"Successfully saved {len(distributed_threshold_results[0.40])} records to {output_path}")

# reasoning_output_path = 'extracted_reasoning_sentences_full.json'
# with open(reasoning_output_path, 'w', encoding='utf-8') as f:
#     json.dump(distributed_threshold_results['reasoning_sentences'], f, indent=4, ensure_ascii=False)

# print(f"Successfully saved {len(distributed_threshold_results['reasoning_sentences'])} reasoning sentences to {reasoning_output_path}")

In [ ]:
# import pyarrow as pa
# import pyarrow.parquet as pq

# # save the triplets (this part worked as it is a list of dicts)
# triplet_table_full = pa.Table.from_pylist(distributed_threshold_results[0.40])
# pq.write_table(triplet_table_full, "extracted_triplets_full.parquet", compression="zstd")

# # fix for reasoning sentences: wrap strings in a dictionary to create a column
# # we transform ["string1", "string2"] into [{"text": "string1"}, {"text": "string2"}]
# reasoning_data_as_dicts = [{"reasoning_prose": s} for s in distributed_threshold_results["reasoning_sentences"]]
# reasoning_table_full = pa.Table.from_pylist(reasoning_data_as_dicts)
# pq.write_table(reasoning_table_full, "extracted_reasoning_sentences_full.parquet", compression="zstd")

# print("Both files successfully saved to Parquet using ZSTD compression.")

In [53]:
import pickle
import json
import torch
import numpy as np
import os

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── 1. all_text_blocks (list of strings — the 3-hour result) ──
with open(f"{CHECKPOINT_DIR}/all_text_blocks.json", "w") as f:
    json.dump(all_text_blocks, f)
print(f"✓ all_text_blocks: {len(all_text_blocks)} windows")

# ── 2. case_metadata_store (complex nested dict — pickle is fastest) ──
with open(f"{CHECKPOINT_DIR}/case_metadata_store.pkl", "wb") as f:
    pickle.dump(case_metadata_store, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"✓ case_metadata_store: {len(case_metadata_store)} cases")

# ── 3. anchor_clause_embeddings (GPU tensor) ──
torch.save(anchor_clause_embeddings.cpu(), f"{CHECKPOINT_DIR}/anchor_clause_embeddings.pt")
print(f"✓ anchor_clause_embeddings: {anchor_clause_embeddings.shape}")

# ── 4. clause_and_terms (list of dicts) ──
# vague_terms is a set — convert to list for JSON serialization
clause_and_terms_serializable = [
    {**item, "vague_terms": list(item["vague_terms"])} for item in clause_and_terms
]
with open(f"{CHECKPOINT_DIR}/clause_and_terms.json", "w") as f:
    json.dump(clause_and_terms_serializable, f)
print(f"✓ clause_and_terms: {len(clause_and_terms)} entries")

print(f"\nAll checkpoints saved to {CHECKPOINT_DIR}/")

✓ all_text_blocks: 16040 windows
✓ case_metadata_store: 6873 cases
✓ anchor_clause_embeddings: torch.Size([1428, 768])
✓ clause_and_terms: 1428 entries

All checkpoints saved to /kaggle/working/checkpoints/


In [ ]:
# To restore in a fresh session
import pickle
import json
import torch

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# ── Restore everything ──
with open(f"{CHECKPOINT_DIR}/all_text_blocks.json", "r") as f:
    all_text_blocks = json.load(f)

with open(f"{CHECKPOINT_DIR}/case_metadata_store.pkl", "rb") as f:
    case_metadata_store = pickle.load(f)

anchor_clause_embeddings = torch.load(
    f"{CHECKPOINT_DIR}/anchor_clause_embeddings.pt",
    weights_only=True
).to("cuda")

with open(f"{CHECKPOINT_DIR}/clause_and_terms.json", "r") as f:
    clause_and_terms = json.load(f)
    for item in clause_and_terms:
        item["vague_terms"] = set(item["vague_terms"])  # restore sets

print(f"Restored: {len(all_text_blocks)} windows, {len(case_metadata_store)} cases, "
      f"{anchor_clause_embeddings.shape} anchors, {len(clause_and_terms)} clauses")